# Tutorial: RAG con un PDF del BOE

Este cuaderno construye un ejemplo de RAG en Python usando el documento local `BOE-A-2015-2870.pdf`.

RAG significa *Retrieval-Augmented Generation*: antes de pedir una respuesta al modelo, recuperamos fragmentos relevantes de una base documental y los usamos como contexto.

Flujo del ejemplo:

1. Leer texto del PDF.
2. Dividir el documento en fragmentos.
3. Crear embeddings de los fragmentos.
4. Buscar los fragmentos mas similares a una pregunta.
5. Responder usando solo el contexto recuperado.

El cuaderno usa el entorno `AIAerospace` y la API de OpenAI.

## 1. Preparacion

Antes de ejecutar el cuaderno, activa el entorno `AIAerospace` y define `OPENAI_API_KEY`.

```powershell
conda activate AIAerospace
$env:OPENAI_API_KEY="sk-..."
jupyter lab
```

El ejemplo usa `pypdf`, `numpy`, `scikit-learn` y `openai`, que estan disponibles en el entorno local.

In [2]:
from openai import OpenAI
from pathlib import Path
from pypdf import PdfReader
from sklearn.metrics.pairwise import cosine_similarity
import json
import numpy as np
import os
import re
import textwrap

PDF_PATH = Path("BOE-A-2015-2870.pdf")
CACHE_PATH = Path("Data/boe_a_2015_2870_embeddings.json")

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

client = OpenAI()

def show(text, width=95):
    print(textwrap.fill(str(text), width=width))

print("Modelo generativo:", MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo: gpt-5.4-nano
Modelo de embeddings: text-embedding-3-small
PDF: C:\Users\igome\OneDrive - Universidad Politécnica de Madrid\Documents\Docencia\AI for Aeroespace\Contenido\Generativa\Programas\LLM\BOE-A-2015-2870.pdf


## 2. Cargar el PDF

Extraemos el texto pagina a pagina. Guardar el numero de pagina en la metadata es importante para poder citar de donde sale cada fragmento.

In [3]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def load_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_index, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_index, "text": text})
    return pages

pages = load_pdf_pages(PDF_PATH)

print(f"Paginas con texto: {len(pages)}")
show(pages[0]["text"][:1200])

Paginas con texto: 7
BOLETÍN OFICIAL DEL ESTADO Núm. 65 Martes 17 de marzo de 2015 Sec. III. Pág. 24131 III. OTRAS
DISPOSICIONES MINISTERIO DE INDUSTRIA, ENERGÍA Y TURISMO 2870 Orden IET/457/2015, de 11 de
marzo, por la que se modifica la Orden IET/786/2013, de 7 de mayo, por la que se establecen las
bases reguladoras de la concesión de ayudas en el ámbito de las tecnologías de la información y
las comunicaciones (TIC) y la sociedad de la información, dentro del Plan de Investigación
Científica y Técnica y de Innovación 2013-2016 en el marco de la acción estratégica de economía
y sociedad digital. La Orden IET/786/2013, de 7 de mayo, estableció las bases reguladoras de la
concesión de ayudas en el ámbito de las tecnologías de la información y las comunicaciones y la
sociedad de la información en el marco de la Acción Estratégica de Economía y Sociedad Digital
y la Agenda Digital para España. De acuerdo con la disposición adicional segunda, las bases
reguladoras están sujetas a la norma

## 3. Dividir el texto en fragmentos

Los modelos de embeddings trabajan mejor con fragmentos de tamano moderado. Si los fragmentos son demasiado grandes, mezclan temas; si son demasiado pequenos, pierden contexto.

Aqui usamos una division simple por palabras con solapamiento.

In [4]:
def chunk_page(page, words_per_chunk=220, overlap=45):
    words = page["text"].split()
    chunks = []
    step = words_per_chunk - overlap

    for start in range(0, len(words), step):
        end = start + words_per_chunk
        chunk_words = words[start:end]
        if len(chunk_words) < 40:
            continue
        chunks.append(
            {
                "page": page["page"],
                "chunk_id": f"p{page['page']}_w{start}",
                "text": " ".join(chunk_words),
            }
        )
    return chunks

chunks = []
for page in pages:
    chunks.extend(chunk_page(page))

print("Numero de fragmentos:", len(chunks))
print("Primer fragmento:")
show(chunks[0]["text"][:900])

Numero de fragmentos: 24
Primer fragmento:
BOLETÍN OFICIAL DEL ESTADO Núm. 65 Martes 17 de marzo de 2015 Sec. III. Pág. 24131 III. OTRAS
DISPOSICIONES MINISTERIO DE INDUSTRIA, ENERGÍA Y TURISMO 2870 Orden IET/457/2015, de 11 de
marzo, por la que se modifica la Orden IET/786/2013, de 7 de mayo, por la que se establecen las
bases reguladoras de la concesión de ayudas en el ámbito de las tecnologías de la información y
las comunicaciones (TIC) y la sociedad de la información, dentro del Plan de Investigación
Científica y Técnica y de Innovación 2013-2016 en el marco de la acción estratégica de economía
y sociedad digital. La Orden IET/786/2013, de 7 de mayo, estableció las bases reguladoras de la
concesión de ayudas en el ámbito de las tecnologías de la información y las comunicaciones y la
sociedad de la información en el marco de la Acción Estratégica de Economía y Sociedad Digital
y la Agenda Digital para España. De acuerdo con l


## 4. Crear embeddings

Un embedding es un vector numerico que representa el significado de un texto. Calculamos un embedding para cada fragmento del BOE.

Para no gastar tokens cada vez que se reinicia el notebook, se guarda una cache en `Data/boe_a_2015_2870_embeddings.json`.

In [5]:
def embed_texts(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
        )
        vectors.extend([item.embedding for item in response.data])
    return vectors

def load_or_create_index(chunks, cache_path):
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        data = json.loads(cache_path.read_text(encoding="utf-8"))
        if data.get("embedding_model") == EMBEDDING_MODEL and len(data.get("chunks", [])) == len(chunks):
            print("Indice cargado desde cache:", cache_path)
            return data["chunks"], np.array(data["embeddings"], dtype=np.float32)

    print("Creando embeddings. Esto consume una llamada a la API.")
    texts = [chunk["text"] for chunk in chunks]
    embeddings = embed_texts(texts)

    data = {
        "embedding_model": EMBEDDING_MODEL,
        "source_pdf": str(PDF_PATH),
        "chunks": chunks,
        "embeddings": embeddings,
    }
    cache_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    print("Indice guardado en:", cache_path)

    return chunks, np.array(embeddings, dtype=np.float32)

indexed_chunks, embeddings = load_or_create_index(chunks, CACHE_PATH)
print("Matriz de embeddings:", embeddings.shape)

Creando embeddings. Esto consume una llamada a la API.
Indice guardado en: Data\boe_a_2015_2870_embeddings.json
Matriz de embeddings: (24, 1536)


## 5. Recuperar fragmentos relevantes

Para una pregunta nueva, calculamos su embedding y buscamos los fragmentos mas parecidos con similitud coseno.

In [6]:
def embed_query(query):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )
    return np.array(response.data[0].embedding, dtype=np.float32).reshape(1, -1)

def retrieve(query, top_k=4):
    query_vector = embed_query(query)
    scores = cosine_similarity(query_vector, embeddings)[0]
    best_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(best_indices, start=1):
        chunk = indexed_chunks[int(idx)]
        results.append({
            "rank": rank,
            "score": float(scores[idx]),
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
        })
    return results

question = "Cual es el objeto de la Orden IET/457/2015?"
retrieved = retrieve(question, top_k=4)

for item in retrieved:
    print(f"[{item['rank']}] pagina {item['page']} | score={item['score']:.3f} | {item['chunk_id']}")
    show(item["text"][:500])
    print()

[1] pagina 1 | score=0.651 | p1_w0
BOLETÍN OFICIAL DEL ESTADO Núm. 65 Martes 17 de marzo de 2015 Sec. III. Pág. 24131 III. OTRAS
DISPOSICIONES MINISTERIO DE INDUSTRIA, ENERGÍA Y TURISMO 2870 Orden IET/457/2015, de 11 de
marzo, por la que se modifica la Orden IET/786/2013, de 7 de mayo, por la que se establecen las
bases reguladoras de la concesión de ayudas en el ámbito de las tecnologías de la información y
las comunicaciones (TIC) y la sociedad de la información, dentro del Plan de Investigación
Científica y Técnica y de Innova

[2] pagina 1 | score=0.624 | p1_w350
n.º 1303/2013 del Parlamento Europeo y del Consejo de 17 de diciembre de 2013, por el que se
establecen disposiciones comunes relativas al Fondo Europeo de Desarrollo Regional, al Fondo
Social Europeo, al Fondo de Cohesión, al Fondo Europeo Agrícola de Desarrollo Rural y al Fondo
Europeo Marítimo y de la Pesca, y por el que se establecen disposiciones generales relativas al
Fondo Europeo de Desarrollo Regional, al Fondo So

## 6. Responder con contexto recuperado

Ahora pasamos los fragmentos recuperados al modelo. La instruccion clave es que responda solo con el contexto y que cite las paginas usadas.

In [7]:
def build_context(retrieved_chunks):
    blocks = []
    for item in retrieved_chunks:
        blocks.append(
            f"[Fuente {item['rank']} | pagina {item['page']} | score {item['score']:.3f}]\n"
            f"{item['text']}"
        )
    return "\n\n".join(blocks)

def answer_with_rag(question, top_k=4):
    retrieved_chunks = retrieve(question, top_k=top_k)
    context = build_context(retrieved_chunks)

    response = client.responses.create(
        model=MODEL,
        instructions=(
            "Eres un asistente docente que explica documentos legales espanoles. "
            "Responde en espanol claro. Usa solo el CONTEXTO proporcionado. "
            "Si el contexto no contiene la respuesta, di: 'No aparece en los fragmentos recuperados'. "
            "Cita siempre las paginas usadas con el formato (pag. X). "
            "No des asesoramiento juridico; esto es un ejemplo docente."
        ),
        input=(
            f"PREGUNTA:\n{question}\n\n"
            f"CONTEXTO RECUPERADO:\n{context}\n\n"
            "RESPUESTA:"
        ),
        max_output_tokens=650,
    )

    return response.output_text, retrieved_chunks

answer, sources = answer_with_rag(question)
show(answer)

print("\nFuentes recuperadas:")
for source in sources:
    print(f"- pagina {source['page']}, {source['chunk_id']}, score={source['score']:.3f}")

El objeto de la **Orden IET/457/2015, de 11 de marzo**, es **modificar la Orden IET/786/2013,
de 7 de mayo**, que establecía las bases reguladoras de la concesión de ayudas en el ámbito de
las **tecnologías de la información y las comunicaciones (TIC) y la sociedad de la
información** (dentro del Plan de Investigación Científica y Técnica y de Innovación 2013-2016,
en el marco de la Acción Estratégica de Economía y Sociedad Digital), para **adaptarla a los
cambios de la normativa comunitaria aplicable al período 2014-2020**. (pág. 1)

Fuentes recuperadas:
- pagina 1, p1_w0, score=0.651
- pagina 1, p1_w350, score=0.624
- pagina 2, p2_w0, score=0.563
- pagina 2, p2_w175, score=0.559


## 7. Probar varias preguntas

Estas preguntas estan pensadas para el documento `BOE-A-2015-2870.pdf`. Cambia las preguntas para observar como varian los fragmentos recuperados.

In [8]:
questions = [
    "Que normativa europea motiva la modificacion de la orden?",
    "Que ayudas o actuaciones se regulan en el documento?",
    "Que cambios se introducen respecto a la Orden IET/786/2013?",
]

for q in questions:
    print("=" * 90)
    print("Pregunta:", q)
    answer, sources = answer_with_rag(q, top_k=4)
    show(answer)
    print("Fuentes:", [(s["page"], round(s["score"], 3)) for s in sources])

Pregunta: Que normativa europea motiva la modificacion de la orden?
La modificación de la Orden se motiva por la **actualización de la normativa comunitaria/UE
aplicable al período 2014-2020**, en concreto:  1) **Reglamento (UE) n.º 651/2014 de la
Comisión, de 17 de junio de 2014** (“Reglamento general de exención por categorías”), por el
que se declaran determinadas categorías de ayudas compatibles con el mercado interior (D.O.U.E.
L187, de 26 de junio de 2014). (pag. 1)  2) **Reglamento (CE) n.º 1303/2013 del Parlamento
Europeo y del Consejo, de 17 de diciembre de 2013**, que establece disposiciones comunes sobre
los Fondos (incluido el FEDER) y deroga el Reglamento (CE) n.º 1083/2006. (pag. 1)  En el
propio texto se indica que “es preciso proceder a la modificación… con el fin de adaptarla a la
nueva normativa comunitaria” y que la normativa comunitaria ha sido modificada para el período
**2014-2020**. (pag. 1)
Fuentes: [(1, 0.628), (1, 0.566), (2, 0.51), (1, 0.508)]
Pregunta: Que a

## 8. Guardrail: no responder fuera del documento

Un RAG debe reconocer cuando no tiene contexto suficiente. Este ejemplo pregunta algo que no deberia estar en el BOE y mantiene la restriccion de responder solo con los fragmentos recuperados.

In [9]:
out_of_scope_question = "Que avion comercial tiene mayor alcance en 2026?"
answer, sources = answer_with_rag(out_of_scope_question, top_k=3)

show(answer)

print("\nFragmentos recuperados, probablemente irrelevantes:")
for source in sources:
    print(f"- pagina {source['page']}, score={source['score']:.3f}")

No aparece en los fragmentos recuperados. (pag. 1; pag. 6)

Fragmentos recuperados, probablemente irrelevantes:
- pagina 6, score=0.223
- pagina 1, score=0.217
- pagina 1, score=0.215


## 9. Inspeccion de similitudes

Para depurar un sistema RAG, no basta con mirar la respuesta final. Conviene inspeccionar si la recuperacion esta trayendo fragmentos realmente relevantes.

In [10]:
def inspect_retrieval(query, top_k=6, preview_chars=350):
    results = retrieve(query, top_k=top_k)
    print("Pregunta:", query)
    print()
    for item in results:
        print(f"Rank {item['rank']} | pag. {item['page']} | score={item['score']:.3f}")
        show(item["text"][:preview_chars])
        print()

inspect_retrieval("Reglamento general de exencion por categorias")

Pregunta: Reglamento general de exencion por categorias

Rank 1 | pag. 5 | score=0.624
BOLETÍN OFICIAL DEL ESTADO Núm. 65 Martes 17 de marzo de 2015 Sec. III. Pág. 24135 Décimo. El
apartado 1 de la disposición adicional segunda queda redactado del siguiente modo: «1. Los
proyectos del presente régimen de ayudas cumplen todas las condiciones del Capítulo I, así como
las disposiciones pertinentes del capítulo III del Reglamento general

Rank 2 | pag. 2 | score=0.601
y 108 del Tratado, publicado en el «Diario Oficial de la Unión Europea», L187, de 26 de junio
de 2014, en adelante, Reglamento general de exención por categorías. En el anexo III se
detallan las condiciones que se deben cumplir para que una PYME se considere empresa en
crisis.» Segundo. El artículo 10.1 queda redactado como sigue: «1. Son empresas

Rank 3 | pag. 4 | score=0.578
aprueba el Reglamento sobre las condiciones básicas para el acceso de las personas con
discapacidad a las tecnologías, productos y servicios relaciona

## 10. Cierre

Ideas principales:

- RAG no entrena el modelo; le proporciona contexto recuperado en tiempo de consulta.
- La calidad depende mucho de la extraccion del texto, el tamano de fragmentos y la busqueda semantica.
- Las respuestas deben citar fragmentos o paginas para facilitar verificacion.
- En documentos legales, el sistema debe evitar conclusiones fuera del texto recuperado y mantener revision humana.
- Para produccion se suele sustituir la matriz local por una base vectorial persistente como FAISS, PostgreSQL con pgvector, Azure AI Search u otro motor de busqueda.